# 02 — Gradient deep dive: what does "gradient" mean in SkillOpt?

In deep learning a gradient is `∂loss/∂param`. SkillOpt's "gradient" is **text**: an LLM-generated reflection over a batch of rollouts, distilled into proposed edits to the skill document. This notebook traces the path:

1. `gradient.reflect` — single-rollout reflection
2. `gradient.aggregate` — batch-level merge
3. The resulting "patch" data structure that flows to the optimizer

No LLM calls — source only.

In [1]:
import inspect
import os

import skillopt
from skillopt import gradient

print("gradient public:", [n for n in dir(gradient) if not n.startswith("_")])
print()
print("source files:")
import skillopt.gradient.aggregate
import skillopt.gradient.reflect

for mod in (skillopt.gradient.reflect, skillopt.gradient.aggregate):
    print(f"  {mod.__name__:<35} {inspect.getfile(mod)}")

gradient public: ['aggregate', 'merge_patches', 'reflect', 'run_minibatch_reflect']

source files:
  skillopt.gradient.reflect           /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/gradient/reflect.py
  skillopt.gradient.aggregate         /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/gradient/aggregate.py


## 1. `reflect` — single-rollout reflection

This is the entry point. Given one rollout's trace + score, produce a critique that names the failure mode.

In [2]:
from skillopt.gradient import reflect as reflect_mod

public_fns = [n for n in dir(reflect_mod) if not n.startswith("_") and callable(getattr(reflect_mod, n))]
print("reflect.py callables:", public_fns[:15])
print()

# Show the main reflection function
for name in ("run_minibatch_reflect", "reflect_single", "reflect"):
    fn = getattr(reflect_mod, name, None)
    if fn and callable(fn) and inspect.isfunction(fn):
        sig = inspect.signature(fn)
        print(f"--- {name}{sig} ---")
        src = inspect.getsource(fn)
        # First 35 lines
        print("\n".join(src.splitlines()[:35]))
        print("..." if len(src.splitlines()) > 35 else "")
        break

reflect.py callables: ['ThreadPoolExecutor', 'as_completed', 'chat_optimizer', 'extract_json', 'fmt_minibatch_trajectories', 'fm
t_trajectory', 'format_meta_skill_context', 'get_payload_items', 'is_full_rewrite_minibatch_mode', 'load_prompt', 'normalize_upd
ate_mode', 'payload_key', 'payload_label', 'run_error_analyst_minibatch', 'run_minibatch_reflect']

--- run_minibatch_reflect(results: 'list[dict]', skill_content: 'str', prediction_dir: 'str', patches_dir: 'str', workers: 'int'
, failure_only: 'bool', minibatch_size: 'int' = 8, edit_budget: 'int' = 4, random_seed: 'int | None' = None, *, error_system: 's
tr | None' = None, success_system: 'str | None' = None, rejection_context: 'str' = '', trajectory_memory_context: 'str' = '', st
ep_buffer_context: 'str' = '', meta_skill_context: 'str' = '', update_mode: 'str' = 'patch') -> 'list[dict | None]' ---
def run_minibatch_reflect(
    results: list[dict],
    skill_content: str,
    prediction_dir: str,
    patches_dir: str,
    workers:

## 2. The reflection prompt template

SkillOpt's "loss function" is the natural-language prompt the optimizer LLM sees. Find it in `prompts/`:

In [3]:
prompts_dir = os.path.join(os.path.dirname(skillopt.__file__), "prompts")
print("prompts dir:", prompts_dir)
print()
prompt_files = sorted(os.listdir(prompts_dir))
for f in prompt_files:
    path = os.path.join(prompts_dir, f)
    size = os.path.getsize(path)
    print(f"  {f:<45} {size:>6} bytes")

prompts dir: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/prompts

  __init__.py                                     1893 bytes
  __pycache__                                       96 bytes
  analyst_error.md                                1951 bytes
  analyst_error_full_rewrite.md                   1258 bytes
  analyst_error_rewrite.md                        1934 bytes
  analyst_success.md                              1646 bytes
  analyst_success_full_rewrite.md                 1219 bytes
  analyst_success_rewrite.md                      1408 bytes
  lr_autonomous.md                                 745 bytes
  merge_failure.md                                1422 bytes
  merge_failure_full_rewrite.md                   1099 bytes
  merge_failure_rewrite.md                        1059 bytes
  merge_final.md                                  1398 bytes
  merge_final_full_rewrite.md                     1053 bytes
  merge_final_rewrite.md                           926 bytes
  merge_succes

In [5]:
# The reflection prompts in SkillOpt are named analyst_error / analyst_success — read those
for target in ("analyst_error.md", "analyst_success.md"):
    text = open(os.path.join(prompts_dir, target)).read()
    print(f"=== {target} ({len(text)} chars) ===")
    print(text[:1200])
    print("\n" + "─" * 70 + "\n")

=== analyst_error.md (1947 chars) ===
You are an expert failure-analysis agent for AI agent tasks.

You will be given MULTIPLE failed agent trajectories from a single minibatch
and the current skill document.
Your job is to identify the most important COMMON failure patterns across
the batch and propose a concise set of skill edits.

## Analysis Process
1. Read ALL trajectories in the minibatch.
2. Identify the most prevalent, systematic failure patterns across them.
3. For each pattern, classify its failure type.
4. Propose skill edits that address the COMMON patterns — not individual edge cases.
5. Edits must be generalizable; do not hardcode task-specific values.
6. Only patch gaps in the skill — do not duplicate existing content.

You will be told the maximum number of edits (the budget L). Produce AT MOST L edits,
focusing on the highest-impact patterns. You may produce fewer if warranted.

Respond ONLY with a valid JSON object (no markdown fences, no extra text):
{
  "batch_size"

## 3. `aggregate` — merging patches into a single edit proposal

After `reflect` runs N minibatches in parallel, the patches must be merged into one coherent edit. That's `aggregate`.

In [6]:
from skillopt.gradient import aggregate as agg_mod

public = [n for n in dir(agg_mod) if not n.startswith("_") and callable(getattr(agg_mod, n))]
print("aggregate.py callables:", public)
print()

merge_fn = getattr(agg_mod, "merge_patches", None)
if merge_fn:
    print(f"signature: {inspect.signature(merge_fn)}\n")
    src = inspect.getsource(merge_fn)
    print(src[:2200] + ("\n..." if len(src) > 2200 else ""))

aggregate.py callables: ['ThreadPoolExecutor', 'as_completed', 'chat_optimizer', 'extract_json', 'format_meta_skill_context', 'g
et_payload_items', 'is_full_rewrite_minibatch_mode', 'is_rewrite_mode', 'load_prompt', 'merge_patches', 'normalize_update_mode',
 'payload_key', 'payload_label']

signature: (skill_content: 'str', failure_patches: 'list[dict]', success_patches: 'list[dict]', batch_size: 'int' = 8, verbose:
'bool' = True, workers: 'int' = 16, update_mode: 'str' = 'patch', meta_skill_context: 'str' = '') -> 'dict'

def merge_patches(
    skill_content: str,
    failure_patches: list[dict],
    success_patches: list[dict],
    batch_size: int = 8,
    verbose: bool = True,
    workers: int = 16,
    update_mode: str = "patch",
    meta_skill_context: str = "",
) -> dict:
    """Failure-first hierarchical merge with support count tracking.

    1. Merge failure patches independently (parallel)
    2. Merge success patches independently (parallel)
    3. Final merge: combine both 

## 4. The "gradient" data structure

What flows from reflect → aggregate → optimizer is a **patch**: a list of `add` / `delete` / `replace` operations against the skill markdown. Look at the type definitions:

In [7]:
from skillopt import types as types_mod

print("file:", inspect.getfile(types_mod))
print()
public = [n for n in dir(types_mod) if not n.startswith("_") and not n[0].islower()]
print("public types:", public)
print()
# Show source of each
src = inspect.getsource(types_mod)
print(src[:3000] + ("\n..." if len(src) > 3000 else ""))

file: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/types.py

public types: ['Any', 'BatchSpec', 'Edit', 'EditOp', 'FailureSummaryEntry', 'GateAction', 'GateResult', 'Literal', 'Patch', 'Raw
Patch', 'RolloutResult', 'SlowUpdateResult']

"""Standardized I/O types for the ReflACT pipeline.

Shared dataclass definitions for the 6-stage per-step pipeline
and the 2 epoch-level stages.  All types support round-trip
conversion to/from plain dicts for incremental adoption.

Re-exports
----------
GateResult, GateAction — from skillopt.evaluation.gate
BatchSpec              — from skillopt.datasets.base
"""
from __future__ import annotations

from dataclasses import dataclass, field, fields as dc_fields
from typing import Any, Literal

from skillopt.evaluation.gate import GateAction, GateResult  # noqa: F401
from skillopt.datasets.base import BatchSpec  # noqa: F401


# ── Atomic types ─────────────────────────────────────────────────────────

EditOp = Literal["append", "insert_after", "repla

## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. `reflect` — single-rollout reflection
- 2. The reflection prompt template
- 3. `aggregate` — merging patches into a single edit proposal
- 4. The "gradient" data structure
- 5. Putting it together — the "backward pass"

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import collections as _c
import json as _json
from pathlib import Path as _Path

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/skillopt/02-gradient-deep-dive.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print("output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")

## 5. Putting it together — the "backward pass"

```
rollout result(s)
     │
     ▼
reflect()  ── per-minibatch LLM call ──▶  patch JSON files on disk
     │                                          │
     │       (N minibatches → N patches)        │
     │                                          ▼
     │                                merge_patches()
     │                                          │
     ▼                                          ▼
optimizer/* (next notebook)        single proposed edit
```

The "gradient" is a **patch dict**, not a tensor. It's persisted to disk (`patches_dir`), then read by the optimizer's edit-acceptance logic.

## Data sources

| Source | Path |
|---|---|
| `gradient.reflect.run_minibatch_reflect` | `skillopt/gradient/reflect.py` |
| `gradient.aggregate.merge_patches` | `skillopt/gradient/aggregate.py` |
| Reflection prompts | `skillopt/prompts/analyst_*.md`, `skillopt/prompts/merge_*.md` |
| Patch type | `skillopt/types.py` |

→ **Next:** [`03-optimizer-types.ipynb`](03-optimizer-types.ipynb).